# Multi-Digit Handwritten Number Recognition using CNN + OpenCV

This project reads a handwritten number (for example, `12345`) from one image.

**How it works**
1. Train a CNN using the MNIST handwritten digit dataset.
2. Upload an image containing a number.
3. OpenCV separates the individual digits from left to right.
4. CNN predicts each digit and combines them into one number.

**Important:** This first version works best when digits are written separately with a small gap. Touching or overlapping digits may not segment correctly, and predictions are not guaranteed to be 100% accurate.


## 1. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from google.colab import files
from PIL import Image
from io import BytesIO

## 2. Load and prepare MNIST dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)

## 3. Build the CNN model

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Train the model

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

## 5. Check model accuracy

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Test accuracy:", round(accuracy * 100, 2), "%")

## 6. Upload a handwritten multi-digit image

Upload a clear image containing one number, such as `12345`.

**For better results:** use dark ink on plain white paper, write digits from left to right, leave a small gap between digits, and avoid tilted or overlapping digits.

In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))

image = Image.open(BytesIO(uploaded[file_name])).convert("RGB")
image = np.array(image)

plt.figure(figsize=(8, 3))
plt.imshow(image)
plt.axis("off")
plt.show()

## 7. Preprocess the image and separate digits

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

# Make the handwritten digits white and the background black
border = np.concatenate([gray[0, :], gray[-1, :], gray[:, 0], gray[:, -1]])
if np.mean(border) < 127:
    gray = 255 - gray

blur = cv2.GaussianBlur(gray, (3, 3), 0)
_, thresh = cv2.threshold(
    blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# Find separate digit regions
contours, _ = cv2.findContours(
    thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

boxes = []
for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    if h >= 8 and w >= 2:
        boxes.append((x, y, w, h))

boxes = sorted(boxes, key=lambda item: item[0])

print("Detected digit regions:", len(boxes))

plt.figure(figsize=(10, 3))
plt.imshow(thresh, cmap="gray")
plt.axis("off")
plt.show()

## 8. Convert each region to MNIST format and predict

In [ ]:
def prepare_digit(thresh, box):
    x, y, w, h = box
    digit = thresh[y:y+h, x:x+w]

    # Keep the original shape and fit inside a 20 x 20 area
    scale = 20 / max(w, h)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    digit = cv2.resize(digit, (new_w, new_h))

    canvas = np.zeros((28, 28), dtype=np.uint8)
    x_start = (28 - new_w) // 2
    y_start = (28 - new_h) // 2
    canvas[y_start:y_start+new_h, x_start:x_start+new_w] = digit

    # Center the digit by its centre of mass
    moments = cv2.moments(canvas)
    if moments["m00"] != 0:
        cx = int(moments["m10"] / moments["m00"])
        cy = int(moments["m01"] / moments["m00"])
        shift_x = 14 - cx
        shift_y = 14 - cy
        matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
        canvas = cv2.warpAffine(canvas, matrix, (28, 28))

    return canvas / 255.0


predicted_digits = []

if len(boxes) == 0:
    print("No digits detected. Try a clearer image.")
else:
    plt.figure(figsize=(2 * len(boxes), 2))

    for i, box in enumerate(boxes):
        digit_image = prepare_digit(thresh, box)
        prediction = model.predict(
            digit_image.reshape(1, 28, 28, 1), verbose=0
        )
        digit = int(np.argmax(prediction))
        confidence = float(np.max(prediction)) * 100
        predicted_digits.append(str(digit))

        plt.subplot(1, len(boxes), i + 1)
        plt.imshow(digit_image, cmap="gray")
        plt.title(f"{digit}\n{confidence:.1f}%")
        plt.axis("off")

    plt.show()

    number = "".join(predicted_digits)
    print("Recognized number:", number)

## 9. Save the trained model

In [ ]:
model.save("multidigit_number_model.keras")
files.download("multidigit_number_model.keras")

## Conclusion

The CNN learns to recognize individual handwritten digits from MNIST. OpenCV finds separate digit regions in the uploaded image. The CNN predicts each region, and the predictions are joined to form the complete number.

**Limitation:** This is a segmentation-based approach. If two digits touch, overlap, or are detected as one region, the final number may be incorrect. Test with clear, separated digits first.
